# Chapitre 3 — Atelier à compléter
Implémenter les fonctions signalées ; conserver le protocole de séparation.


## Étape 1
Consigner les versions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
print('NumPy :', np.__version__)



## Étape 2
Dessiner les rôles avant tout entraînement.


In [ ]:
# CELL 2 — Des rôles disjoints, avant toute sélection.
def make_data(n, seed, correlation=.9):
    r = np.random.default_rng(seed)
    signal = r.normal(size=(n, 2))
    latent = (signal[:, 0]*signal[:, 1] > 0).astype(float)
    nuisance = r.normal(size=(n, 8))
    same = r.random(n) < correlation
    shortcut = np.where(same, 2*latent-1, -(2*latent-1)) + r.normal(0,.2,n)
    X = np.column_stack([signal,nuisance,shortcut])
    # Même bruit de cible indépendant dans chaque partition.
    y = np.where(r.random(n)<.1, 1-latent, latent).reshape(-1,1)
    return X, y

raw = {name: make_data(n, seed) for name,n,seed in
       [('train',96,10),('validation',192,20),('calibration',192,30),('test',1000,40)]}
raw['shift'] = make_data(1000,50,correlation=.1)
mu, sigma = raw['train'][0].mean(0), raw['train'][0].std(0)
data = {name: ((X-mu)/sigma,y) for name,(X,y) in raw.items()}
Xtr,ytr = data['train']; Xv,yv = data['validation']
print({name: X.shape for name,(X,y) in data.items()})



## Étape 3
Vérifier les gradients avec un masque dropout fixé.


In [ ]:
# CELL 3 — Dropout inversé, L2 explicite et perte prédictive séparée.
def sigmoid(z): return np.exp(-np.logaddexp(0.,-z))
def init(seed=0,width=64):
    r=np.random.default_rng(seed)
    return {'W1':r.normal(0,1/np.sqrt(Xtr.shape[1]),(Xtr.shape[1],width)),
            'b1':np.zeros((1,width)), 'W2':r.normal(0,1/np.sqrt(width),(width,1)),
            'b2':np.zeros((1,1))}

def forward(X,p,drop=0.,rng=None):
    # TODO : tanh, masque inversé et cache du passage avant.
    raise NotImplementedError("Implémenter forward")

def loss_grad(X,y,p,l2=0.,drop=0.,rng=None):
    # TODO : NLL, pénalité poids et gradients, avec le même masque.
    raise NotImplementedError("Implémenter loss_grad")

def evaluate(X,y,p):
    z,_=forward(X,p)  # Dropout désactivé pour toutes les comparaisons.
    return float(np.mean(np.logaddexp(0.,z)-y*z)),float(((z>=0)==y).mean())



## Étape 4
Prédire le checkpoint rendu et distinguer durée/époque retenue.


In [ ]:
# CELL 4 — Sélection de checkpoint sans consulter le test.
def fit(seed=0,l2=0.,drop=0.,early=False,augment=False,epochs=400):
    p=init(seed); m={k:np.zeros_like(v) for k,v in p.items()}
    v={k:np.zeros_like(a) for k,a in p.items()}
    order_rng=np.random.default_rng(100+seed)
    drop_rng=np.random.default_rng(200+seed)
    aug_rng=np.random.default_rng(300+seed)
    best=copy.deepcopy(p); best_value=np.inf; best_epoch=0; stale=0; t=0; hist=[]
    for epoch in range(epochs):
        order=order_rng.permutation(len(Xtr))
        for start in range(0,len(order),32):
            ids=order[start:start+32]; xb=Xtr[ids].copy(); yb=ytr[ids]
            if augment:
                # Réflexion simultanée des deux coordonnées brutes : classe XOR inchangée.
                xb=xb*sigma+mu
                flip=aug_rng.random(len(xb))<.5
                xb[flip,:2]*=-1
                xb=(xb-mu)/sigma
            _,g=loss_grad(xb,yb,p,l2,drop,drop_rng); t+=1
            for k in p:
                m[k]=.9*m[k]+.1*g[k]; v[k]=.999*v[k]+.001*g[k]**2
                p[k]-=.005*(m[k]/(1-.9**t))/(np.sqrt(v[k]/(1-.999**t))+1e-8)
        train_nll,train_acc=evaluate(Xtr,ytr,p); val_nll,val_acc=evaluate(Xv,yv,p)
        hist.append(dict(epoch=epoch+1,steps=t,train_nll=train_nll,val_nll=val_nll,
                         train_acc=train_acc,val_acc=val_acc))
        if val_nll<best_value-1e-4:
            best_value=val_nll; best_epoch=epoch+1; best=copy.deepcopy(p); stale=0
        else: stale+=1
        if early and stale>=30:break
    returned=best if early else p
    return returned,hist,dict(epochs_run=len(hist),steps=t,
                             returned_epoch=best_epoch if early else len(hist))



## Étape 5
Comparer les procédures, pas seulement les scores gagnants.


In [ ]:
# CELL 5 — Ablations, graines appariées, hyperparamètres annoncés.
configs={'reference':{},'L2':{'l2':.001},'dropout':{'drop':.25},
         'early':{'early':True},'augmentation':{'augment':True}}
runs={}
for name,cfg in configs.items():
    for seed in range(3):runs[name,seed]=fit(seed=seed,**cfg)
mean_validation={}
for name in configs:
    scores=[evaluate(Xv,yv,runs[name,s][0])[0] for s in range(3)]
    mean_validation[name]=float(np.mean(scores))
    print(name,'NLL validation moyenne ± écart-type :',np.mean(scores),np.std(scores),
          'budget graine 0 :',runs[name,0][2])
selected=min(mean_validation,key=mean_validation.get)
# La graine 0 est fixée pour le modèle illustratif, pas choisie après les scores.
model=runs[selected,0][0]
print('Procédure choisie sur validation :',selected,'; graine fixée : 0')
fig,ax=plt.subplots(1,2,figsize=(10,4))
for name in configs:
    h=runs[name,0][1]
    ax[0].plot([r['steps'] for r in h],[r['train_nll'] for r in h],label=name)
    ax[1].plot([r['steps'] for r in h],[r['val_nll'] for r in h],label=name)
ax[0].set(title='Train, mode évaluation',xlabel='Mises à jour',ylabel='NLL prédictive')
ax[1].set(title='Validation, checkpoint courant',xlabel='Mises à jour',ylabel='NLL prédictive')
for a in ax:a.legend()
plt.tight_layout();plt.show()



## Étape 6
Définir Brier et ECE sans ambiguïté.


In [ ]:
# CELL 6 — Scores probabilistes et diagramme de fiabilité top-label binaire.
def nll_logits(z,y):return float(np.mean(np.logaddexp(0.,z)-y*z))
def reliability(z,y,bins=10):
    # TODO : confiance top-label, bins non vides, effectifs et ECE.
    raise NotImplementedError("Implémenter reliability")

def metrics(z,y):
    p=sigmoid(z);rows,ece=reliability(z,y)
    return dict(accuracy=float(((z>=0)==y).mean()),nll=nll_logits(z,y),
                brier=float(np.mean((p-y)**2)),ece=ece)



## Étape 7
Ajuster T uniquement sur calibration, puis geler toute décision.


In [ ]:
# CELL 7 — Température ajustée sur calibration seulement ; test final ensuite.
Xcal,ycal=data['calibration'];zcal,_=forward(Xcal,model)
temperatures=np.unique(np.r_[1.,np.geomspace(.2,5.,101)])
T=float(min(temperatures,key=lambda t:nll_logits(zcal/t,ycal)))
assert nll_logits(zcal/T,ycal)<=nll_logits(zcal,ycal)+1e-12
print('Température choisie sur calibration :',T)
frozen_results={}
for name in ['test','shift']:
    X,y=data[name];z,_=forward(X,model)
    before,after=metrics(z,y),metrics(z/T,y)
    assert before['accuracy']==after['accuracy']
    frozen_results[name]=(z.copy(),y.copy(),before,after)
    print(name,'avant',before,'après',after)
    rows,_=reliability(z/T,y)
    plt.plot([r[1] for r in rows],[r[2] for r in rows],'o-',label=name)
plt.plot([.5,1],[.5,1],'k--',label='Référence parfaite')
plt.xlabel('Confiance moyenne');plt.ylabel('Taux de bonnes réponses');plt.legend();plt.show()
# Aucun réajustement de T ni choix de modèle après lecture de ces résultats.



## Étape 8
Déclarer les hypothèses de l’intervalle conditionnel aux modèles.


In [ ]:
# CELL 8 — Bootstrap apparié, modèles et test fixés.
z,y,_,_=frozen_results['test'];truth=y.ravel()
z_ref,_=forward(data['test'][0],runs['reference',0][0])
diff=((z.ravel()>=0)==truth).astype(float)-((z_ref.ravel()>=0)==truth).astype(float)
r=np.random.default_rng(900)
boot=np.array([diff[r.integers(0,len(diff),len(diff))].mean() for _ in range(1000)])
print('Différence accuracy sélection − référence :',diff.mean(),
      '; IC bootstrap percentile 95% :',np.quantile(boot,[.025,.975]))
# Cet intervalle conditionne sur les modèles appris et suppose les lignes i.i.d.
# Il n'intègre ni les réentraînements ni l'incertitude du choix de procédure.



## Étape 9
Vérifier l’unité d’indépendance avant de conclure.


In [ ]:
# CELL 9 — Fuite entre groupes : isoler le protocole avec un 1-plus-proche-voisin.
r=np.random.default_rng(123);groups=120;repeats=6;dim=20
centers=r.normal(size=(groups,dim));group_y=r.integers(0,2,groups)
ids=np.repeat(np.arange(groups),repeats)
GX=centers[ids]+r.normal(0,.03,(len(ids),dim));Gy=group_y[ids]
perm=r.permutation(len(GX));cut=int(.8*len(perm))
row_train,row_test=perm[:cut],perm[cut:]
gperm=r.permutation(groups);gtrain=set(gperm[:96])
group_train=np.where(np.isin(ids,list(gtrain)))[0]
group_test=np.where(~np.isin(ids,list(gtrain)))[0]
assert not set(ids[group_train])&set(ids[group_test])
def nn_accuracy(a,b):
    distances=((GX[b,None,:]-GX[None,a,:])**2).sum(2)
    pred=Gy[a[np.argmin(distances,axis=1)]]
    return float((pred==Gy[b]).mean())
print('Groupes partagés, séparation par lignes :',len(set(ids[row_train])&set(ids[row_test])))
print('Accuracy par lignes :',nn_accuracy(row_train,row_test))
print('Accuracy nouveaux groupes :',nn_accuracy(group_train,group_test))
# Ici les étiquettes par groupe sont aléatoires : reconnaître une identité ne généralise pas.



## Étape 10
Expliquer l’optimisme malgré un risque identique.


In [ ]:
# CELL 10 — Optimisme de sélection, simulation à risque identique.
r=np.random.default_rng(321);true_risk=.25;nval=100
for candidates in [1,5,20,100]:
    # Modèle stylisé : erreurs d'estimation indépendantes entre candidats.
    estimates=r.binomial(nval,true_risk,size=(500,candidates))/nval
    winners=estimates.min(1)
    print(candidates,'candidats ; erreur validation gagnante moyenne :',winners.mean(),
          '; vrai risque de tous les candidats :',true_risk)
# Ce schéma n'affirme pas que les scores réels sur un même jeu sont indépendants.
